# Train models

In [1]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import numpy as np
import os
import itertools
import subprocess
import time

In [2]:
dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

In [3]:
def is_job_running(job_name):
    result = subprocess.run(['squeue', '-o', '%.28i %.28j %.28u %R', '-u', 'kemal.inecik'],  capture_output=True, text=True)
    jobs = [[j.strip() for j in i.split()]for i in result.stdout.strip().split('\n')]
    for jobid, jobname, jobuser, jobnode in jobs:
        if job_name == jobname:
            return True
    return False

```
#SBATCH -J {job_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

```
#SBATCH -J {job_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 32
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 1-23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

In [4]:
step_per_epoch = 841922 * 0.25 * 0.8 / 512
print(step_per_epoch)

epochs = list(itertools.chain(range(20, 52), range(52, 100, 2), range(100, 200, 10), range(200, 421, 20)))
epochs = list(np.arange(0, 20, 0.10)) + epochs
epochs = [round(i, 2) for i in epochs]
steps = [int(i*step_per_epoch) for i in epochs]
np.random.shuffle(steps)

print(np.array(steps))
print(len(steps))

328.87578125000005
[  3354   6281     32   6445   8221  15786  16114  28283    559  31572
  17101   4998   4012  17759  12497   5360   5229   1151   2236   5262
  72352   5689   8550   1381  13812   3584  15128    822    131   4900
   6544  11181  30914   2400   7564  62486   6413   4110   3420   3157
   4078   6215   2992  36176   5031   6182  25652   3288   5196   6511
   2071   1743   5426  16772  28941   1512   2696  10524   3617   3091
   2335   4538     65     98   3255   1479   5327   8879  26967   2006
   4834   5919  39465   3683   3058   5558 138127  12168   4275   2663
   3749  32887   2762  18417   9866    295   1216   2828   1808   4735
   1907    756  49331    493   5788  29598  14799  19074  13155    197
  19732   5097  24336   1874    723   5590   5623   4143   4308   3716
   6051    361   1775    460   5130   3124   2170    230  59197   3387
  14470   3814   4670   5163   4374   9208  14141  21048  26310   4209
   1545   3847   1973      0   2959   5821   3321   6084  

In [5]:
job_count = 0
overwrite = False
print(f" - Number of models to be trained: {len(steps)!r}")

cpu_gpu = "gpu"

for model_str in ["scanvi", "scvi"]:
    
    for epoch in steps:
        
        output_dir_path = os.path.join(dataset_dir, f"model_suo_incremental_training_{model_str}_epoch_{epoch}")
        log_file = os.path.join(logs_directory, f"slurm_out_model_suo_incremental_training_{cpu_gpu}_{model_str}_epoch_{epoch}.log")
        job_name = f"incr_{cpu_gpu}_{model_str}_{epoch}"
        python_name = f"model_training.py"

        if is_job_running(job_name):
            print(f"Training {job_name!r} on {cpu_gpu!r} keeps going for model {model_str!r} and for epoch {epoch!r}.")
        elif overwrite or not os.path.exists(output_dir_path) or not os.path.isdir(output_dir_path):
            try:
                slurm_script = f"""#!/bin/bash
#SBATCH -J {job_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

source activate sctram_dev_env
python -u {os.path.join(helpers_directory, python_name)} --epoch "{epoch}" --model_str "{model_str}" --overwrite "{overwrite}"
    """
                script_name = os.path.join(logs_directory, f"slurm_job_model_suo_incremental_training_{cpu_gpu}_{model_str}_epoch_{epoch}.sh")
                with open(script_name, "w") as f:
                    f.write(slurm_script)

                print(f"Submitted job on {cpu_gpu!r} {job_count+1} on {cpu_gpu!r}: {job_name!r} for model {model_str!r} and for epoch {epoch!r}")
                subprocess.run(["sbatch", script_name])
                job_count += 1
            finally:
                # time.sleep(0.05)
                os.remove(script_name)

            assert is_job_running(job_name), f"Training {job_name!r} on {cpu_gpu!r} error: model {model_str!r} and for epoch {epoch!r}."
        else:
            print(f"Model exists for model {model_str!r} and for epoch {epoch!r}")
    # if job_count > 32:
    #     break

print(f" - Number of jobs submitted: {job_count}")

 - Number of models to be trained: 278
Model exists for model 'scanvi' and for epoch 3354
Model exists for model 'scanvi' and for epoch 6281
Model exists for model 'scanvi' and for epoch 32
Model exists for model 'scanvi' and for epoch 6445
Model exists for model 'scanvi' and for epoch 8221
Model exists for model 'scanvi' and for epoch 15786
Model exists for model 'scanvi' and for epoch 16114
Model exists for model 'scanvi' and for epoch 28283
Model exists for model 'scanvi' and for epoch 559
Model exists for model 'scanvi' and for epoch 31572
Model exists for model 'scanvi' and for epoch 17101
Model exists for model 'scanvi' and for epoch 4998
Model exists for model 'scanvi' and for epoch 4012
Model exists for model 'scanvi' and for epoch 17759
Model exists for model 'scanvi' and for epoch 12497
Model exists for model 'scanvi' and for epoch 5360
Model exists for model 'scanvi' and for epoch 5229
Model exists for model 'scanvi' and for epoch 1151
Model exists for model 'scanvi' and for

In [6]:
1

1